# 🦜🔗 LangChain Complete Tutorial
## From Fundamentals to Real Applications

---

### What You Will Learn
| Section | Topic |
|---|---|
| 1 | Models — calling GPT with `ChatOpenAI` |
| 2 | Prompts — `ChatPromptTemplate`, few-shot |
| 3 | Output Parsers — Str, JSON, Pydantic |
| 4 | LCEL — building chains with `|` operator |
| 5 | Memory — multi-turn conversations |
| 6 | RAG — answer questions from your own documents |
| 7 | Agents — AI that decides which tools to use |
| 8 | Real Projects — end-to-end applications |

---

### LangChain Architecture

```
YOUR APPLICATION
      │
      ▼
┌─────────────────────────────────────────┐
│              LANGCHAIN                  │
│                                         │
│  Prompts → Models → Output Parsers      │
│       └──── LCEL chains (|) ────┘       │
│                                         │
│  Memory    Retrievers    Agents+Tools   │
└─────────────────────────────────────────┘
      │
      ▼
  OpenAI / Anthropic / Local LLM
```

**Java analogy:** LangChain is like Spring Boot for AI — it provides structure,
dependency injection (swappable models), and patterns for building AI applications
without writing all the plumbing code yourself.

In [ ]:
# Install all required packages
# Run this cell once, then restart kernel if needed
!pip install langchain langchain-openai langchain-core langchain-community \
             langchain-text-splitters faiss-cpu tiktoken pydantic \
             langchainhub -q

In [ ]:
from dotenv import load_dotenv
import os

# override=True ensures the latest key from .env is always used,
# even if the kernel already has an old key cached in memory
load_dotenv(override=True)

print("Key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

---
## Section 1 — Models: The AI Brain

### What is a Model in LangChain?
A model is the AI that processes your input and returns a response.
LangChain wraps raw API calls so you can:
- Switch between GPT, Claude, Gemini with **one line change**
- Add streaming, retries, logging automatically
- Chain models into pipelines

### Two Types
| Type | Input | Output | Example |
|---|---|---|---|
| **LLM** | plain string | plain string | older API style |
| **ChatModel** | list of messages | message | modern — what we use |

### Message Types
```python
SystemMessage  → sets the AI's role/context (like a job description)
HumanMessage   → what the user says
AIMessage      → what the AI responded (stored in conversation history)
```

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Create the model
# Java analogy: ChatOpenAI llm = new ChatOpenAI(model, temperature)
llm = ChatOpenAI(
    model="gpt-4o",    # which GPT version to use
    temperature=0.7,   # creativity: 0=focused/deterministic, 1=creative/varied
    max_tokens=500     # max length of response
)

# --- Basic invocation with messages ---
# Pass a list of messages representing the conversation
messages = [
    # SystemMessage tells the AI who it is and how to behave
    SystemMessage(content="You are a helpful Python tutor. Be concise."),
    # HumanMessage is the user's question
    HumanMessage(content="What is a list comprehension? Give one example.")
]

# .invoke() sends the messages and returns an AIMessage object
response = llm.invoke(messages)

print("Response type:", type(response).__name__)  # AIMessage
print("Content:")
print(response.content)

# response.response_metadata contains token usage, model name, etc.
print("\nTokens used:", response.response_metadata.get("token_usage", {}))

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o", temperature=0.7)

# --- Streaming Output ---
# Instead of waiting for the FULL response, receive tokens as they are generated
# This is how ChatGPT shows text appearing word-by-word in the browser
# Use case: chatbots where instant feedback improves user experience

print("Streaming response (appears token by token):")
print("-" * 50)

# .stream() returns a generator — yields chunks of the response
for chunk in llm.stream([HumanMessage(content="Explain Python decorators in 3 sentences.")]):
    # chunk.content is a small piece of the response (a few tokens)
    print(chunk.content, end="", flush=True)  # end="" prevents newlines between chunks

print("\n" + "-" * 50)
print("✅ Streaming complete")

# --- Temperature Experiment ---
# Same question, 3 different temperatures → different styles of answers
print("\n🌡️  Temperature comparison:")
question = [HumanMessage(content="Give a one-sentence tagline for a coffee shop.")]

for temp in [0.0, 0.7, 1.5]:
    model = ChatOpenAI(model="gpt-4o", temperature=temp)
    result = model.invoke(question)
    print(f"  temp={temp}: {result.content}")

---
## Section 2 — Prompts: Structured Instructions

### What is a Prompt Template?
Instead of hardcoding the full prompt every time, you define a **template with placeholders**
and fill in the values at runtime.

**Java analogy:** `PreparedStatement` in JDBC — define the structure once,
bind different values each time. Reusable, safe, clean.

```python
# Without template (bad — hardcoded for one use)
prompt = "Explain recursion to a beginner"

# With template (good — reusable for any topic and level)
template = "Explain {topic} to a {level} developer"
```

### Types of Prompt Templates
| Class | Use Case |
|---|---|
| `PromptTemplate` | Simple text prompts with variables |
| `ChatPromptTemplate` | Chat-style (system + human + ai messages) |
| `FewShotChatMessagePromptTemplate` | Include examples to guide output style |

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o", temperature=0.7)

# --- ChatPromptTemplate (most commonly used) ---
# from_messages() takes a list of (role, template_string) tuples
# {variable_name} placeholders are filled in when you invoke
prompt = ChatPromptTemplate.from_messages([
    # System message: sets the AI's expertise and behavior
    ("system", "You are an expert {domain} instructor. Explain clearly with analogies."),
    # Human message: the actual task with variable slots
    ("human", "Explain {concept} in simple terms. Include a real-world analogy.")
])

# Build chain: prompt fills variables → llm calls API → parser extracts text
chain = prompt | llm | StrOutputParser()

# Invoke with values for {domain} and {concept}
result = chain.invoke({"domain": "machine learning", "concept": "overfitting"})
print("Result 1 — ML overfitting:")
print(result)
print()

# SAME chain, DIFFERENT inputs — no rebuilding needed
# This is the power of templates: one setup, infinite variations
result2 = chain.invoke({"domain": "software engineering", "concept": "SOLID principles"})
print("Result 2 — SOLID principles:")
print(result2[:300] + "...")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o", temperature=0)

# --- Few-Shot Prompting ---
# Provide EXAMPLES of the input-output pairs you expect
# The AI learns the pattern from examples and applies it to new inputs
# Use case: enforce specific output format, tone, or structure

# Step 1: Define the format for each example (human input + ai output)
example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

# Step 2: Provide the actual examples
# These teach the AI the format you want: 'TERM: one-line definition'
examples = [
    {"input": "What is an API?",
     "output": "API: A contract that lets two software systems communicate."},
    {"input": "What is a database?",
     "output": "DATABASE: Organized storage for structured data with fast retrieval."},
    {"input": "What is caching?",
     "output": "CACHING: Storing frequently used data in fast memory to avoid re-computation."}
]

# Step 3: Assemble the few-shot prompt template
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

# Step 4: Build the full prompt — system + examples + new question
final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise tech dictionary. Follow the format from examples exactly."),
    few_shot_prompt,    # inject all examples here
    ("human", "{question}")
])

chain = final_prompt | llm | StrOutputParser()

# The AI follows the TERM: definition format because of the examples
for q in ["What is recursion?", "What is load balancing?"]:
    print(chain.invoke({"question": q}))

---
## Section 3 — Output Parsers: Transform AI Responses

The AI returns an `AIMessage` object. Output parsers transform it into the exact
Python type your application needs.

```
AIMessage → StrOutputParser    → str (plain text)
AIMessage → JsonOutputParser   → dict (Python dictionary)
AIMessage → PydanticOutputParser → MyModel instance (typed object)
```

**Java analogy:** Like `ObjectMapper.readValue(json, MyClass.class)` in Jackson —
deserializing AI output into a typed object.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o", temperature=0.7)

# --- StrOutputParser ---
# Simplest parser: extracts .content from AIMessage → plain Python string
# Use when you just need the text and don't need structure

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a creative writer."),
    ("human", "Write a 2-line poem about {subject}.")
])

# Without parser: chain returns AIMessage object
chain_no_parser = prompt | llm
raw = chain_no_parser.invoke({"subject": "Java programming"})
print("Without parser — type:", type(raw).__name__)  # AIMessage
print("Without parser — value:", raw)                # full AIMessage object
print()

# With StrOutputParser: chain returns plain string
chain_with_parser = prompt | llm | StrOutputParser()
text = chain_with_parser.invoke({"subject": "Java programming"})
print("With parser — type:", type(text).__name__)  # str
print("With parser — value:", text)                # just the text

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

llm = ChatOpenAI(model="gpt-4o", temperature=0)

# --- JsonOutputParser ---
# Parses AI response as JSON → Python dict
# Use when you need structured data from unstructured text
# Real-world use case: extract fields from customer emails, CVs, receipts

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a data extractor. Always respond with valid JSON only. No explanation."),
    ("human", """
Extract information from this text.
Return JSON with these exact keys: name, age, city, job, skill_level.

Text: {text}
""")
])

# JsonOutputParser automatically parses JSON string into Python dict
chain = prompt | llm | JsonOutputParser()

result = chain.invoke({
    "text": "Hi, I'm Priya Sharma, a 28-year-old senior software engineer "
            "from Bangalore. I specialize in Python and have 5 years of experience."
})

print("Result type:", type(result).__name__)  # dict
print("Result:", result)
print()
# Access like a regular Python dict
print(f"Name: {result['name']}")
print(f"City: {result['city']}")
print(f"Job:  {result['job']}")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

llm = ChatOpenAI(model="gpt-4o", temperature=0)

# --- PydanticOutputParser ---
# Parse AI output into a TYPED Python object (like a Java POJO / record)
# Pydantic validates types automatically — you get type safety + IDE autocomplete
# Best practice for production applications

# Step 1: Define your data model with Pydantic
# Field(description=...) tells the AI what each field should contain
class JobListing(BaseModel):
    title: str = Field(description="Job title")
    company: str = Field(description="Company name")
    location: str = Field(description="City or remote")
    salary_range: str = Field(description="Salary range if mentioned, else 'Not mentioned'")
    required_skills: List[str] = Field(description="List of required technical skills")
    experience_years: int = Field(description="Minimum years of experience required")

# Step 2: Create parser from your model
parser = PydanticOutputParser(pydantic_object=JobListing)

# Step 3: Build prompt — include format instructions so AI knows exact schema
# .partial() pre-fills a variable so you don't need to pass it at invoke time
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a job listing extractor.\n{format_instructions}"),
    ("human", "Extract job details from:\n{listing}")
]).partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser

# Step 4: Extract from real job listing text
listing_text = """
Senior Python Developer at TechCorp Bangalore
We are looking for an experienced Python developer with 4+ years of experience.
Required skills: Python, Django, PostgreSQL, Docker, AWS.
Salary: 18-25 LPA. Hybrid work model (Whitefield office).
"""

job = chain.invoke({"listing": listing_text})

# job is a JobListing instance — fully typed, IDE autocomplete works!
print(f"Title:    {job.title}")
print(f"Company:  {job.company}")
print(f"Location: {job.location}")
print(f"Salary:   {job.salary_range}")
print(f"Skills:   {job.required_skills}")
print(f"Exp:      {job.experience_years}+ years")
print(f"\nType check: {type(job).__name__}")  # JobListing

---
## Section 4 — LCEL: LangChain Expression Language

LCEL is the modern way to build LangChain pipelines using the `|` (pipe) operator.

```python
chain = prompt | llm | output_parser
result = chain.invoke({"variable": "value"})
```

**Java analogy:** Java Streams API — `list.stream().filter(...).map(...).collect(...)`
Each step transforms the data and passes it to the next.

### LCEL Runnable Components
| Component | Purpose |
|---|---|
| `|` operator | Chain components: A → B → C |
| `RunnablePassthrough` | Pass input through unchanged |
| `RunnableParallel` | Run multiple chains simultaneously |
| `RunnableLambda` | Wrap any Python function as a chain step |
| `.stream()` | Stream tokens as they arrive |
| `.batch()` | Process multiple inputs at once |

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda

llm = ChatOpenAI(model="gpt-4o", temperature=0.7)
parser = StrOutputParser()

# ==========================================================
# RunnablePassthrough — pass input unchanged to the next step
# Use case: you need the original input in the final output
# ==========================================================
translate_prompt = ChatPromptTemplate.from_template("Translate to French: {text}")

# {"original": input, "translation": run the chain}
passthrough_chain = RunnableParallel(
    original=RunnablePassthrough(),          # keep original input
    translation=translate_prompt | llm | parser   # also translate it
)

result = passthrough_chain.invoke("How are you doing today?")
print("RunnablePassthrough example:")
print(f"  Original:    {result['original']}")
print(f"  Translation: {result['translation']}")
print()

# ==========================================================
# RunnableParallel — run MULTIPLE chains at the SAME TIME
# All chains execute simultaneously (parallel API calls)
# Results are combined into a dict
# Java analogy: CompletableFuture.allOf(...) — parallel async execution
# ==========================================================
formal_prompt   = ChatPromptTemplate.from_template("Rewrite very formally: {text}")
casual_prompt   = ChatPromptTemplate.from_template("Rewrite for Gen-Z casual style: {text}")
summary_prompt  = ChatPromptTemplate.from_template("Summarize in 5 words: {text}")

# All three chains run in PARALLEL — same input, three different outputs
parallel_chain = RunnableParallel(
    formal  = formal_prompt  | llm | parser,
    casual  = casual_prompt  | llm | parser,
    summary = summary_prompt | llm | parser
)

results = parallel_chain.invoke({
    "text": "Please be informed the project deadline has been moved to next Friday."
})

print("RunnableParallel example (3 chains ran simultaneously):")
print(f"  Formal:  {results['formal']}")
print(f"  Casual:  {results['casual']}")
print(f"  Summary: {results['summary']}")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

llm = ChatOpenAI(model="gpt-4o", temperature=0)
parser = StrOutputParser()

# ==========================================================
# RunnableLambda — wrap any Python function as a chain step
# Use when you need custom preprocessing/postprocessing
# Java analogy: Function<T, R> in Java Streams — map(fn)
# ==========================================================
def preprocess_input(text: str) -> dict:
    """Clean and enrich the input before sending to AI"""
    from datetime import date
    cleaned = text.strip().lower()
    word_count = len(text.split())
    return {
        "text": cleaned,
        "word_count": word_count,
        "date": str(date.today())
    }

prompt = ChatPromptTemplate.from_template(
    "Summarize this {word_count}-word text in one sentence (as of {date}):\n{text}"
)

# RunnableLambda makes a regular function chainable with |
chain = RunnableLambda(preprocess_input) | prompt | llm | parser

result = chain.invoke(
    "Machine learning is a branch of AI that allows computers to learn "
    "from data and improve performance without being explicitly programmed."
)
print("RunnableLambda example:")
print(result)
print()

# ==========================================================
# .batch() — process multiple inputs EFFICIENTLY
# Sends all inputs at once (parallel API calls internally)
# Much faster than calling .invoke() in a loop
# ==========================================================
translate_chain = ChatPromptTemplate.from_template("Translate to Spanish: {text}") | llm | parser

inputs = [
    {"text": "Good morning"},
    {"text": "Thank you very much"},
    {"text": "I love programming"},
]

# .batch() processes all inputs and returns a list of results
translations = translate_chain.batch(inputs)

print(".batch() example:")
for inp, trans in zip(inputs, translations):
    print(f"  EN: {inp['text']}  →  ES: {trans}")

---
## Section 5 — Memory: Multi-Turn Conversations

### The Problem
By default, each `llm.invoke()` call is **stateless** — the model has no memory
of previous questions. Every call starts fresh.

### The Solution
Manually maintain a **list of messages** (the conversation history) and pass
the entire list on every call. The model "remembers" because it sees all prior turns.

```
Turn 1: [System] + [Human: Q1]                    → AI gives A1
Turn 2: [System] + [Human: Q1] + [AI: A1] + [Human: Q2]  → AI gives A2
Turn 3: [System] + [all previous] + [Human: Q3]   → AI gives A3
```

### Token Management Problem
Long conversations accumulate many messages → many tokens → expensive and slow.

**Solution:** Sliding window — keep only the last N turns.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

llm = ChatOpenAI(model="gpt-4o", temperature=0.3)

# -------------------------------------------------------
# Pattern: Maintain conversation history as a list
# The list grows with each turn — this IS the memory
# -------------------------------------------------------

# System message defines the AI's role for the entire conversation
SYSTEM = SystemMessage(content=(
    "You are a helpful Python tutor. Keep answers short. "
    "Always refer to previous answers when relevant."
))

# Start with just the system message
history = [SYSTEM]

def chat(user_input: str, max_turns: int = 5) -> str:
    """Send a message, maintaining a sliding window of conversation history."""

    # Step 1: Append the user's message
    history.append(HumanMessage(content=user_input))

    # Step 2: Sliding window — keep system + last max_turns exchanges
    # Each exchange = 1 HumanMessage + 1 AIMessage = 2 messages
    # Without this, history grows forever → token limit errors
    if len(history) > (max_turns * 2 + 1):  # +1 for system message
        system_msg = history[0]                   # always keep system
        recent = history[-(max_turns * 2):]       # last N turns
        trimmed_history = [system_msg] + recent
    else:
        trimmed_history = history

    # Step 3: Send FULL history to model — this is how memory works
    response = llm.invoke(trimmed_history)

    # Step 4: Store AI response for next turn's context
    history.append(AIMessage(content=response.content))

    print(f"  [History: {len(trimmed_history)} msgs sent to API]")
    return response.content

# Multi-turn conversation — each question builds on previous
print("=" * 60)

print("You: What is a Python list?")
print("AI:", chat("What is a Python list?"))
print()

print("You: How is it different from a tuple?")
print("AI:", chat("How is it different from a tuple?"))
print()

# AI should reference 'list' and 'tuple' because it has the full history
print("You: Can you show code comparing both?")
print("AI:", chat("Can you show code comparing both?"))
print()

print(f"Total messages stored in history: {len(history)}")

---
## Section 6 — RAG: Answer Questions From Your Own Documents

### What is RAG (Retrieval-Augmented Generation)?
GPT doesn't know your company's internal documents, recent events, or private data.
RAG gives the AI access to your specific documents **without retraining the model**.

```
User question
     │
     ▼
Find relevant chunks   ← Vector Database (similarity search)
     │
     ▼
Inject into prompt     ← "Here is relevant context: ..."
     │
     ▼
LLM generates answer   ← Grounded in YOUR documents
```

### Steps to Build RAG
1. **Load** documents (PDF, web page, text file)
2. **Split** into chunks (text splitter)
3. **Embed** chunks → convert to vectors (OpenAIEmbeddings)
4. **Store** vectors in a database (FAISS)
5. **Retrieve** relevant chunks for a query
6. **Generate** answer using retrieved context + LLM

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken

# --- Text Splitting ---
# Problem: Large documents exceed the LLM's context window limit
# Solution: Split into smaller overlapping chunks
# 
# RecursiveCharacterTextSplitter splits in this order (tries each separator):
#   1. \n\n  (paragraph breaks — best split point)
#   2. \n    (line breaks)
#   3. .     (sentence ends)
#   4. " "   (words — last resort)
# This preserves natural text boundaries as much as possible

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,      # max characters per chunk
    chunk_overlap=50,    # overlap between chunks — preserves context at boundaries
    length_function=len  # measure by character count
)

document = """
Chapter 1: Introduction to Transformers

The Transformer architecture was introduced in the 2017 paper 'Attention is All You Need'
by Vaswani et al. It replaced recurrent neural networks (RNNs) for sequence modeling tasks.

The key innovation is the self-attention mechanism, which allows the model to weigh
the importance of different words in the input when producing each output token.
Unlike RNNs, Transformers process all tokens in parallel, making training much faster.

Chapter 2: BERT and GPT

BERT (Bidirectional Encoder Representations from Transformers) uses only the encoder
part of the Transformer. It reads text in both directions and is used for classification
and understanding tasks.

GPT (Generative Pre-trained Transformer) uses only the decoder part. It reads text
left-to-right and is optimized for text generation tasks.
""" * 2  # repeat to simulate larger document

chunks = splitter.split_text(document)

print(f"Document: {len(document)} chars")
print(f"Chunks:   {len(chunks)} chunks")
print()

# Count tokens in each chunk using tiktoken
encoder = tiktoken.get_encoding("cl100k_base")  # GPT-4 tokenizer
for i, chunk in enumerate(chunks[:4]):
    tokens = len(encoder.encode(chunk))
    print(f"Chunk {i+1}: {len(chunk)} chars, {tokens} tokens")
    print(f"  Preview: {chunk[:80]}...")
    print()

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ============================================================
# FULL RAG PIPELINE
# Use case: Internal company knowledge base chatbot
# ============================================================

llm = ChatOpenAI(model="gpt-4o", temperature=0)

# --- STEP 1: Create knowledge base documents ---
# In production: load from PDFs, databases, web pages
# Document has page_content (text) + metadata (source info)
docs = [
    Document(page_content="TechNova was founded in 2019 by Arjun Mehta and Priya Singh in Bangalore.",
             metadata={"source": "company_history"}),
    Document(page_content="TechNova products: CloudDash (monitoring), DataPulse (analytics), SecureID (auth).",
             metadata={"source": "products"}),
    Document(page_content="Leave policy: 15 earned leave, 12 casual, 8 sick leave days per year.",
             metadata={"source": "hr_policy"}),
    Document(page_content="WFH policy: up to 3 days per week with manager approval.",
             metadata={"source": "hr_policy"}),
    Document(page_content="TechNova raised Series B funding of $25M in 2023 from Sequoia Capital.",
             metadata={"source": "funding"}),
    Document(page_content="Office locations: Bangalore (HQ), Mumbai, Singapore. Total staff: 250.",
             metadata={"source": "company_info"}),
]

# --- STEP 2: Embed and store ---
# OpenAIEmbeddings converts text to 1536-dim vectors
# FAISS stores them and enables fast similarity search (no server needed)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(docs, embeddings)
print(f"✅ Vector store: {len(docs)} documents embedded")

# --- STEP 3: Create retriever ---
# Retriever finds the k most similar documents for any query
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# --- STEP 4: RAG prompt ---
# Inject retrieved context into the prompt
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a helpful company knowledge assistant.
Answer ONLY from the provided context. If not in context, say so.

Context:
{context}
"""),
    ("human", "{question}")
])

def format_docs(documents):
    """Join retrieved documents into a single context string"""
    return "\n\n".join(
        f"[{doc.metadata['source']}]: {doc.page_content}" for doc in documents
    )

# --- STEP 5: Build RAG chain ---
# retriever fetches relevant docs → format_docs joins them → inject as {context}
# question passes through unchanged via RunnablePassthrough
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# --- STEP 6: Test ---
questions = [
    "Who founded TechNova?",
    "How many WFH days can I take?",
    "What was TechNova's last funding round?",
    "What is TechNova's annual revenue?",   # NOT in docs — should say so
]

print()
for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_chain.invoke(q)}")
    print()

---
## Section 7 — Agents: AI That Decides What To Do

### What is an Agent?
A chain executes a **fixed sequence**: A → B → C, always.
An agent **decides dynamically** what to do based on the situation.

```
Chain:  Input → Step1 → Step2 → Output  (predetermined)
Agent:  Input → Think → Decide tool → Use tool → Think → ... → Answer
```

### ReAct Pattern (Reasoning + Acting)
The most common agent pattern. Each loop:
```
Thought:     "I need to check the order status first"
Action:      call get_order_status(order_id="ORD001")
Observation: "Shipped — Expected delivery tomorrow"
Thought:     "Now I have enough info to answer"
Final Answer: "Your order ORD001 has shipped and arrives tomorrow!"
```

### Tools
Functions decorated with `@tool` that the agent can call.
The AI reads the docstring to understand when and how to use each tool.

In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_react_agent, AgentExecutor
from langchain import hub

llm = ChatOpenAI(model="gpt-4o", temperature=0)

# -------------------------------------------------------
# Define Tools using @tool decorator
# The DOCSTRING is critical — the AI reads it to decide when to use each tool
# -------------------------------------------------------

@tool
def get_order_status(order_id: str) -> str:
    """Look up the current status of a customer order by order ID.
    Use this when a customer asks about their order, delivery, or shipment."""
    # In production: query your orders database here
    orders = {
        "ORD001": "Shipped — Estimated delivery: Tomorrow by 6 PM",
        "ORD002": "Processing — Will ship in 1-2 business days",
        "ORD003": "Delivered — Delivered on May 12 at 2:30 PM",
        "ORD004": "Cancelled — Refund of ₹2,500 processed",
    }
    return orders.get(order_id, f"Order '{order_id}' not found. Please verify the order ID.")

@tool
def check_product_availability(product_name: str) -> str:
    """Check if a product is in stock and get its current price.
    Use this when a customer asks about product availability or pricing."""
    inventory = {
        "laptop": "In stock — Dell Inspiron 15 — ₹65,000",
        "phone": "In stock — Samsung Galaxy S24 — ₹79,000",
        "headphones": "Out of stock — Sony WH-1000XM5 — Expected restock: June 1",
        "tablet": "In stock — iPad Air — ₹58,000",
    }
    key = product_name.lower()
    for k, v in inventory.items():
        if k in key:
            return v
    return f"Product '{product_name}' not found in inventory system."

@tool
def raise_support_ticket(order_id: str, issue: str) -> str:
    """Create a customer support ticket for an order issue.
    Use this when a customer reports a problem that needs human follow-up."""
    # In production: create a ticket in your CRM (Freshdesk, Zendesk, etc.)
    import hashlib
    ticket_id = "TKT-" + hashlib.md5(f"{order_id}{issue}".encode()).hexdigest()[:6].upper()
    return (
        f"Support ticket {ticket_id} created for order {order_id}.\n"
        f"Issue: {issue}\n"
        f"Our team will respond within 24 hours via email."
    )

tools = [get_order_status, check_product_availability, raise_support_ticket]

# Pull the standard ReAct prompt from LangChain Hub
# This prompt is pre-built to guide the Thought/Action/Observation loop
react_prompt = hub.pull("hwchase17/react")

# Create the ReAct agent — combines LLM + tools + prompt
agent = create_react_agent(llm, tools, react_prompt)

# AgentExecutor manages the Thought→Action→Observation loop
executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,            # print each Thought/Action/Observation step
    max_iterations=6,        # stop if loop runs more than 6 times (safety)
    handle_parsing_errors=True  # recover gracefully if AI output is malformed
)

print("=" * 60)
print("Customer Support Agent")
print("=" * 60)

# The agent will decide WHICH tools to call and in what order
result = executor.invoke({
    "input": "Hi! My order ORD001 shows shipped but I haven't received it. "
             "Also, is the Sony headphone available? If order is an issue, please raise a ticket."
})

print("\nFinal Answer:")
print(result["output"])

---
## Section 8 — Real Project: Production Support Chatbot

Combining everything:
- **Memory** for multi-turn conversation
- **RAG** for policy knowledge
- **Function calling** for structured extraction
- **Prompt engineering** for consistent responses

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o", temperature=0.3)

# Production-quality system prompt
# This is the single most impactful thing for chatbot quality
SYSTEM_PROMPT = """You are a customer support agent for ShopEasy, India's leading e-commerce platform.

PERSONALITY: Warm, empathetic, solution-focused. Use first name when known.

POLICIES YOU MUST FOLLOW:
- Returns: Accepted within 30 days, free pickup
- Refunds: 5-7 business days after return received
- Delivery: Standard 2-5 days, Express 1 day (₹99 extra)
- Damaged items: Offer immediate replacement OR full refund, no questions asked
- Escalation: If unresolved in 2 turns, offer to connect to specialist team

RESPONSE FORMAT:
1. Acknowledge the issue with empathy (1 sentence)
2. Provide the solution clearly
3. Ask if there's anything else they need

NEVER: Make up order details. NEVER promise specific delivery times without checking."""

# Conversation history — grows with each turn
conversation = [SystemMessage(content=SYSTEM_PROMPT)]

def support_chat(user_message: str) -> str:
    """Handle a customer message with full conversation context"""
    conversation.append(HumanMessage(content=user_message))

    # Sliding window: keep system + last 10 messages
    ctx = [conversation[0]] + conversation[-10:] if len(conversation) > 11 else conversation

    response = llm.invoke(ctx)
    conversation.append(AIMessage(content=response.content))
    return response.content

# -------------------------------------------------------
# Simulate a realistic customer support session
# -------------------------------------------------------
interactions = [
    "Hi, I received a completely broken laptop from my order #ORD-5789.",
    "I want a replacement, not a refund.",
    "How long will the replacement take to arrive?",
]

print("🛒 ShopEasy Customer Support")
print("=" * 60)

for user_msg in interactions:
    print(f"\nCustomer: {user_msg}")
    reply = support_chat(user_msg)
    print(f"Agent:    {reply}")
    print("-" * 60)

print(f"\nConversation turns: {(len(conversation) - 1) // 2}")

---
## ✅ LangChain Summary

| Component | What It Does | Key Class |
|---|---|---|
| **Models** | Call the AI | `ChatOpenAI` |
| **Prompts** | Structure the input | `ChatPromptTemplate` |
| **Output Parsers** | Transform output | `StrOutputParser`, `PydanticOutputParser` |
| **LCEL** | Connect components | `\|` operator |
| **Memory** | Remember conversation | Manual message list |
| **Text Splitters** | Break large docs | `RecursiveCharacterTextSplitter` |
| **Embeddings** | Text → vectors | `OpenAIEmbeddings` |
| **Vector Stores** | Store + search | `FAISS` |
| **RAG** | Answer from your docs | retriever + rag_chain |
| **Agents** | AI chooses actions | `AgentExecutor` |

## The Core LangChain Pattern
```python
# 1. Define
chain = prompt | llm | output_parser

# 2. Run once
result = chain.invoke({"variable": "value"})

# 3. Run many
results = chain.batch([{"variable": v} for v in values])

# 4. Run streaming
for chunk in chain.stream({"variable": "value"}):
    print(chunk, end="")
```

## 🚀 Next
For **loops, branching, state, and multi-agent workflows** →
See `06_langgraph_complete_tutorial.ipynb`